# Vaccine Hesitancy Prediction — Decision Tree Classifier

> **Research:** Uncovering determinants of vaccine hesitancy in India: A comparative machine learning framework for data-driven insights  
> **Conference:** Young Researchers' Conference 2025, Dept. of CSE (AI&ML), PCCOE

This notebook implements a **Decision Tree Classifier** to predict vaccine hesitancy from structured survey data collected from Indian respondents.

Decision Trees are the most interpretable model in this study — they produce a visual flowchart of decision rules, making it easy to explain *why* a prediction was made. This is especially valuable for communicating findings to public health policymakers.

---

**Pipeline:**
1. Load & inspect the raw survey data
2. Preprocessing (drop irrelevant columns, encode categoricals)
3. SMOTE class balancing
4. Train-test split
5. Decision Tree training
6. Evaluation (Accuracy, Classification Report, Confusion Matrix)
7. Feature Importance visualization
8. Tree structure visualization

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE

print("All libraries loaded successfully.")

## 2. Load & Inspect Data

The dataset is a primary survey collected from Indian respondents via Google Forms. It is **fully anonymous** — no personally identifiable information was collected.

In [ ]:
df = pd.read_csv("../data/Vaccine_Hesitancy_Survey_Raw.csv")
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(" -", col)

## 3. Preprocessing

We drop columns that are irrelevant for modelling:
- **Timestamp** — just a form submission time
- **Consent** — everyone answered Yes (required to submit)
- **Free-text columns** (open-ended answers) — not suitable for this model
- **Cancer vaccine columns** — out of scope for this study

The target variable is: `Did you ever feel hesitant before taking the vaccine?`  
We binarize it into **Hesitant** (Yes / A little) vs **Not Hesitant** (No) for cleaner classification.

In [ ]:
# Drop irrelevant columns: Timestamp, Consent, free-text, cancer vaccine columns
cols_to_drop = [
    "Timestamp",
    "I am 18+ and consent to anonymous use of my answers for research",
    "In your own words, what is your biggest concern or hesitation about vaccines?",
    "Is there anything else you would like to share about your thoughts or experiences related to vaccination?",
    "If a new safe and approved vaccine for cancer (developed anywhere) were available, how likely would you be to take it",
    "If you are unlikely to take a new cancer vaccine, what are your main reasons? (select all that apply)"
]
# Drop by partial match to handle whitespace variations in column names
drop_mask = df.columns.str.contains('|'.join([
    'Timestamp', 'consent', 'own words', 'anything else', 'cancer'
]), case=False)
df = df.loc[:, ~drop_mask]

print(f"Columns after dropping irrelevant ones: {df.shape[1]}")
print(df.columns.tolist())

In [ ]:
# Define target column
target_col = [c for c in df.columns if 'hesitant before taking' in c.lower()][0]
print(f"Target column: '{target_col}'")
print(f"\nOriginal value counts:\n{df[target_col].value_counts()}")

# Binarize: 'No' -> 0 (Not Hesitant), everything else -> 1 (Hesitant)
df['Vaccine_Hesitant'] = df[target_col].apply(lambda x: 0 if str(x).strip().lower() == 'no' else 1)
print(f"\nBinarized target:\n{df['Vaccine_Hesitant'].value_counts()}")

# Separate features and target
X = df.drop(columns=[target_col, 'Vaccine_Hesitant'])
y = df['Vaccine_Hesitant']

In [ ]:
# One-hot encode all categorical features
X_encoded = pd.get_dummies(X, drop_first=False).astype(int)
feature_names = X_encoded.columns.tolist()

print(f"Features after encoding: {X_encoded.shape[1]}")

## 4. Handle Class Imbalance with SMOTE

With only 139 survey responses, class imbalance is a real concern. SMOTE generates synthetic samples for the minority class so the model doesn't just learn to predict the majority.

In [ ]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_encoded, y)

print(f"Before SMOTE: {dict(y.value_counts())}")
print(f"After SMOTE:  {dict(pd.Series(y_resampled).value_counts())}")

## 5. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")

## 6. Train Decision Tree

We use `max_depth=5` to prevent overfitting — unconstrained trees will perfectly memorize training data but generalize poorly. `class_weight='balanced'` further helps with any residual imbalance.

In [ ]:
clf = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    class_weight="balanced",
    random_state=42
)
clf.fit(X_train, y_train)

print("Decision Tree trained successfully.")
print(f"Tree depth: {clf.get_depth()} | Leaves: {clf.get_n_leaves()}")

## 7. Evaluation

In [ ]:
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Not Hesitant", "Hesitant"]))

## 8. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    clf, X_test, y_test,
    display_labels=["Not Hesitant", "Hesitant"],
    cmap="Oranges",
    ax=ax
)
plt.title("Confusion Matrix — Decision Tree", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("../results/confusion_matrix_dt.png", dpi=150)
plt.show()

## 9. Feature Importance

Which features does the tree rely on most when making splits?

In [ ]:
importances = pd.Series(clf.feature_importances_, index=feature_names)
top15 = importances.nlargest(15)

fig, ax = plt.subplots(figsize=(10, 6))
top15.sort_values().plot.barh(ax=ax, color="darkorange")
ax.set_title("Decision Tree — Top 15 Feature Importances", fontsize=13)
ax.set_xlabel("Gini Importance")
plt.tight_layout()
plt.savefig("../results/feature_importance_dt.png", dpi=150)
plt.show()

print("\nTop 15 features:")
print(top15.sort_values(ascending=False).to_string())

## 10. Tree Visualization

One of the biggest advantages of a Decision Tree — you can actually *see* the decision rules.

In [ ]:
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    clf,
    max_depth=3,                        # show top 3 levels for readability
    feature_names=feature_names,
    class_names=["Not Hesitant", "Hesitant"],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
plt.title("Decision Tree — Top 3 Levels", fontsize=15, pad=15)
plt.tight_layout()
plt.savefig("../results/decision_tree_plot.png", dpi=120, bbox_inches="tight")
plt.show()

---

## Summary

| Metric | Value |
|--------|-------|
| Accuracy | ~69.8% |
| Precision | 0.69 |
| Recall | 0.70 |
| F1-Score | 0.69 |

**Observation:** The Decision Tree is the weakest individual model in this study, prone to overfitting even with `max_depth` constrained. However, it provides the most **human-readable decision rules**, making it useful for explaining the key splits to non-technical stakeholders.

The ensemble methods (Random Forest, XGBoost) in `random_forest_xgboost.ipynb` overcome this limitation by averaging over many trees, achieving significantly higher accuracy.